In [25]:
import pandas as pd
import yaml
import sys
from pathlib import Path
import joblib
import numpy as np
from sklearn.model_selection import train_test_split
from typing import Tuple, Any
import warnings
warnings.filterwarnings('ignore')

# define project root
PROJECT_ROOT = Path().resolve().parent

# **Load Config**

In [26]:
def load_config(config_path: str) -> dict:
    """
    Load the configuration file.

    Parameters:
    -----------
    config_path (str): The path to the configuration file.

    Returns:
    --------
    config (dict): The loaded configuration as a dictionary.
    """
    try:
        with open(config_path, 'r') as file:
            config = yaml.safe_load(file)
        if config is None:
            raise ValueError("Configuration file is empty.")
        
    except FileNotFoundError:
        raise FileNotFoundError(f"Configuration file not found at: {config_path}")
    
    except yaml.YAMLError as e:
        raise ValueError(f"Error parsing YAML file: {e}")
    
    return config

# **Update Config**

In [27]:
def update_config(key: str, value: str, config_path: str) -> dict:
    """
    Update a specific key-value pair in the config file.
    
    Parameters:
    -----------
    key (str): The key to update in the config file.
    value (str): The new value to set for the specified key.
    config_path (str): The path to the configuration file.

    Returns:
    --------
    config (dict): The updated configuration as a dictionary.
    """

    # load config
    config = load_config(config_path)
    config[key] = value

    # open config file and write the updated config
    with open(config_path, 'w') as file:
        yaml.dump(config, file)
    
    print(f"Config updated: {key} set to {value}")

    # reload config
    return load_config(config_path)

In [28]:
# test the config load
config = load_config(config_path=PROJECT_ROOT / 'config' / 'config.yaml')
config

{'ALL_COLS': ['Age',
  'Attrition',
  'BusinessTravel',
  'DailyRate',
  'Department',
  'DistanceFromHome',
  'Education',
  'EducationField',
  'EnvironmentSatisfaction',
  'Gender',
  'HourlyRate',
  'JobInvolvement',
  'JobLevel',
  'JobRole',
  'JobSatisfaction',
  'MaritalStatus',
  'MonthlyIncome',
  'MonthlyRate',
  'NumCompaniesWorked',
  'OverTime',
  'PercentSalaryHike',
  'PerformanceRating',
  'RelationshipSatisfaction',
  'StockOptionLevel',
  'TotalWorkingYears',
  'TrainingTimesLastYear',
  'WorkLifeBalance',
  'YearsAtCompany',
  'YearsInCurrentRole',
  'YearsSinceLastPromotion',
  'YearsWithCurrManager'],
 'CAT_COLS': ['BusinessTravel',
  'Department',
  'EducationField',
  'Gender',
  'JobRole',
  'MaritalStatus',
  'OverTime'],
 'NUM_COLS': ['Age',
  'DailyRate',
  'DistanceFromHome',
  'Education',
  'EnvironmentSatisfaction',
  'HourlyRate',
  'JobInvolvement',
  'JobLevel',
  'JobSatisfaction',
  'MonthlyIncome',
  'MonthlyRate',
  'NumCompaniesWorked',
  'Percen

# **About Dataset**

About Dataset

Uncover the factors that lead to employee attrition and explore important questions such as ‘show me a breakdown of distance from home by job role and attrition’ or ‘compare average monthly income by education and attrition’. This is a fictional data set created by IBM data scientists.

Education
- 1 = 'Below College'
- 2 = 'College'
- 3 = 'Bachelor'
- 4 = 'Master'
- 5 = 'Doctor'

EnvironmentSatisfaction
- 1 = 'Low'
- 2 = 'Medium'
- 3 = 'High'
- 4 = 'Very High'

JobInvolvement
- 1 = 'Low'
- 2 = 'Medium'
- 3 = 'High'
- 4 = 'Very High'

JobSatisfaction
- 1 = 'Low'
- 2 = 'Medium'
- 3 = 'High'
- 4 = 'Very High'

PerformanceRating
- 1 = 'Low'
- 2 = 'Good'
- 3 = 'Excellent'
- 4 = 'Outstanding'

RelationshipSatisfaction
- 1 = 'Low'
- 2 = 'Medium'
- 3 = 'High'
- 4 = 'Very High'

WorkLifeBalance
- 1 = 'Bad'
- 2 = 'Good'
- 3 = 'Better'
- 4 = 'Best'

# **Load Data**

In [29]:
def load_data(data_path: str) -> pd.DataFrame:
    """ 
    Load the raw data from the path specified in the config.

    Parameters:
    -----------
    data_path (str): The path to the raw data file.

    Returns:
    --------
    df (pd.DataFrame): The loaded raw data as a pandas DataFrame.
    """

    # make the validation layers 
    try:
        df = pd.read_csv(data_path)
        if df.empty:
            raise ValueError("The raw data file is empty. Please check the file content.")
        else:
            print(f"Raw data loaded successfully from: {data_path}")
            print(f"Data shape: {df.shape}")

    except Exception as e:
        raise ValueError(f"Error loading raw data: {e}")

    return df

In [30]:
data = load_data(data_path=PROJECT_ROOT / 'data' / 'raw' / config['PATH_RAW_DATA'])
data.head()

Raw data loaded successfully from: /home/bagaskoroah/employee_attrition/data/raw/attrition_raw.csv
Data shape: (1470, 35)


,Age,Attrition,BusinessTravel,DailyRate,Department,DistanceFromHome,Education,EducationField,EmployeeCount,EmployeeNumber,...,RelationshipSatisfaction,StandardHours,StockOptionLevel,TotalWorkingYears,TrainingTimesLastYear,WorkLifeBalance,YearsAtCompany,YearsInCurrentRole,YearsSinceLastPromotion,YearsWithCurrManager
0,41,Yes,Travel_Rarely,1102,Sales,1,2,Life Sciences,1,1,...,1,80,0,8,0,1,6,4,0,5
1,49,No,Travel_Frequently,279,Research & Development,8,1,Life Sciences,1,2,...,4,80,1,10,3,3,10,7,1,7
2,37,Yes,Travel_Rarely,1373,Research & Development,2,2,Other,1,4,...,2,80,0,7,3,3,0,0,0,0
3,33,No,Travel_Frequently,1392,Research & Development,3,4,Life Sciences,1,5,...,3,80,0,8,3,3,8,7,3,0
4,27,No,Travel_Rarely,591,Research & Development,2,1,Medical,1,7,...,4,80,1,6,3,3,2,2,2,2


In [31]:
# encode attrition to biner numbers
data['Attrition'] = data['Attrition'].replace({
    'No': 0,
    'Yes': 1
})

data.head()

,Age,Attrition,BusinessTravel,DailyRate,Department,DistanceFromHome,Education,EducationField,EmployeeCount,EmployeeNumber,...,RelationshipSatisfaction,StandardHours,StockOptionLevel,TotalWorkingYears,TrainingTimesLastYear,WorkLifeBalance,YearsAtCompany,YearsInCurrentRole,YearsSinceLastPromotion,YearsWithCurrManager
0,41,1,Travel_Rarely,1102,Sales,1,2,Life Sciences,1,1,...,1,80,0,8,0,1,6,4,0,5
1,49,0,Travel_Frequently,279,Research & Development,8,1,Life Sciences,1,2,...,4,80,1,10,3,3,10,7,1,7
2,37,1,Travel_Rarely,1373,Research & Development,2,2,Other,1,4,...,2,80,0,7,3,3,0,0,0,0
3,33,0,Travel_Frequently,1392,Research & Development,3,4,Life Sciences,1,5,...,3,80,0,8,3,3,8,7,3,0
4,27,0,Travel_Rarely,591,Research & Development,2,1,Medical,1,7,...,4,80,1,6,3,3,2,2,2,2


In [32]:
# check duplicates
data.duplicated().sum()

np.int64(0)

# **Data Validation**

In [33]:
data.dtypes

Age                          int64
Attrition                    int64
BusinessTravel              object
DailyRate                    int64
Department                  object
DistanceFromHome             int64
Education                    int64
EducationField              object
EmployeeCount                int64
EmployeeNumber               int64
EnvironmentSatisfaction      int64
Gender                      object
HourlyRate                   int64
JobInvolvement               int64
JobLevel                     int64
JobRole                     object
JobSatisfaction              int64
MaritalStatus               object
MonthlyIncome                int64
MonthlyRate                  int64
NumCompaniesWorked           int64
Over18                      object
OverTime                    object
PercentSalaryHike            int64
PerformanceRating            int64
RelationshipSatisfaction     int64
StandardHours                int64
StockOptionLevel             int64
TotalWorkingYears   

In [34]:
# change education column to categorical
data['Education'] = data['Education'].astype('object')
data['Education'] = data['Education'].replace({
    1: 'Below College',
    2: 'College',
    3: 'Bachelor',
    4: 'Master',
    5: 'Doctor'
})

data.dtypes

Age                          int64
Attrition                    int64
BusinessTravel              object
DailyRate                    int64
Department                  object
DistanceFromHome             int64
Education                   object
EducationField              object
EmployeeCount                int64
EmployeeNumber               int64
EnvironmentSatisfaction      int64
Gender                      object
HourlyRate                   int64
JobInvolvement               int64
JobLevel                     int64
JobRole                     object
JobSatisfaction              int64
MaritalStatus               object
MonthlyIncome                int64
MonthlyRate                  int64
NumCompaniesWorked           int64
Over18                      object
OverTime                    object
PercentSalaryHike            int64
PerformanceRating            int64
RelationshipSatisfaction     int64
StandardHours                int64
StockOptionLevel             int64
TotalWorkingYears   

In [35]:
# sanity check on dataframe
data.head()

,Age,Attrition,BusinessTravel,DailyRate,Department,DistanceFromHome,Education,EducationField,EmployeeCount,EmployeeNumber,...,RelationshipSatisfaction,StandardHours,StockOptionLevel,TotalWorkingYears,TrainingTimesLastYear,WorkLifeBalance,YearsAtCompany,YearsInCurrentRole,YearsSinceLastPromotion,YearsWithCurrManager
0,41,1,Travel_Rarely,1102,Sales,1,College,Life Sciences,1,1,...,1,80,0,8,0,1,6,4,0,5
1,49,0,Travel_Frequently,279,Research & Development,8,Below College,Life Sciences,1,2,...,4,80,1,10,3,3,10,7,1,7
2,37,1,Travel_Rarely,1373,Research & Development,2,College,Other,1,4,...,2,80,0,7,3,3,0,0,0,0
3,33,0,Travel_Frequently,1392,Research & Development,3,Master,Life Sciences,1,5,...,3,80,0,8,3,3,8,7,3,0
4,27,0,Travel_Rarely,591,Research & Development,2,Below College,Medical,1,7,...,4,80,1,6,3,3,2,2,2,2


In [36]:
# drop irrelevant columns
data = data.drop(columns=['EmployeeCount', 'EmployeeNumber', 'Over18', 'StandardHours'])

# sanity check on dataframe
print('Data shape after dropping irrelevant columns:', data.shape)
data.head()

Data shape after dropping irrelevant columns: (1470, 31)


,Age,Attrition,BusinessTravel,DailyRate,Department,DistanceFromHome,Education,EducationField,EnvironmentSatisfaction,Gender,...,PerformanceRating,RelationshipSatisfaction,StockOptionLevel,TotalWorkingYears,TrainingTimesLastYear,WorkLifeBalance,YearsAtCompany,YearsInCurrentRole,YearsSinceLastPromotion,YearsWithCurrManager
0,41,1,Travel_Rarely,1102,Sales,1,College,Life Sciences,2,Female,...,3,1,0,8,0,1,6,4,0,5
1,49,0,Travel_Frequently,279,Research & Development,8,Below College,Life Sciences,3,Male,...,4,4,1,10,3,3,10,7,1,7
2,37,1,Travel_Rarely,1373,Research & Development,2,College,Other,4,Male,...,3,2,0,7,3,3,0,0,0,0
3,33,0,Travel_Frequently,1392,Research & Development,3,Master,Life Sciences,4,Female,...,3,3,0,8,3,3,8,7,3,0
4,27,0,Travel_Rarely,591,Research & Development,2,Below College,Medical,1,Male,...,3,4,1,6,3,3,2,2,2,2


In [37]:
for col in data.columns.tolist():
    print('Col:', col)
    print(data[col].value_counts(normalize=True))
    print('')

Col: Age
Age
35    0.053061
34    0.052381
36    0.046939
31    0.046939
29    0.046259
32    0.041497
30    0.040816
38    0.039456
33    0.039456
40    0.038776
37    0.034014
28    0.032653
27    0.032653
42    0.031293
39    0.028571
45    0.027891
41    0.027211
26    0.026531
44    0.022449
46    0.022449
43    0.021769
50    0.020408
25    0.017687
24    0.017687
49    0.016327
47    0.016327
55    0.014966
48    0.012925
51    0.012925
53    0.012925
54    0.012245
52    0.012245
22    0.010884
56    0.009524
58    0.009524
23    0.009524
21    0.008844
20    0.007483
59    0.006803
19    0.006122
18    0.005442
60    0.003401
57    0.002721
Name: proportion, dtype: float64

Col: Attrition
Attrition
0    0.838776
1    0.161224
Name: proportion, dtype: float64

Col: BusinessTravel
BusinessTravel
Travel_Rarely        0.709524
Travel_Frequently    0.188435
Non-Travel           0.102041
Name: proportion, dtype: float64

Col: DailyRate
DailyRate
691     0.004082
1082    0.003401
329

In [38]:
# separate each column by data type
num_cols = data.select_dtypes(include=['int64', 'float64']).columns
obj_cols = set(data.columns).difference(set(num_cols))

config = update_config(key='ALL_COLS', value=data.columns.tolist(), config_path=PROJECT_ROOT / 'config' / 'config.yaml')
config = update_config(key='TARGET_COL', value='Attrition', config_path=PROJECT_ROOT / 'config' / 'config.yaml')

Config updated: ALL_COLS set to ['Age', 'Attrition', 'BusinessTravel', 'DailyRate', 'Department', 'DistanceFromHome', 'Education', 'EducationField', 'EnvironmentSatisfaction', 'Gender', 'HourlyRate', 'JobInvolvement', 'JobLevel', 'JobRole', 'JobSatisfaction', 'MaritalStatus', 'MonthlyIncome', 'MonthlyRate', 'NumCompaniesWorked', 'OverTime', 'PercentSalaryHike', 'PerformanceRating', 'RelationshipSatisfaction', 'StockOptionLevel', 'TotalWorkingYears', 'TrainingTimesLastYear', 'WorkLifeBalance', 'YearsAtCompany', 'YearsInCurrentRole', 'YearsSinceLastPromotion', 'YearsWithCurrManager']
Config updated: TARGET_COL set to Attrition


In [39]:
# update range of each numeric columns
keys = [f'RANGE_{col.upper()}' for col in num_cols]

for col, key in zip(num_cols, keys):
    config = update_config(
        key=key,
        value=[int(np.min(data[col])), int(np.max(data[col]))],
        config_path=PROJECT_ROOT / 'config' / 'config.yaml'
    )

Config updated: RANGE_AGE set to [18, 60]
Config updated: RANGE_ATTRITION set to [0, 1]
Config updated: RANGE_DAILYRATE set to [102, 1499]


Config updated: RANGE_DISTANCEFROMHOME set to [1, 29]
Config updated: RANGE_ENVIRONMENTSATISFACTION set to [1, 4]
Config updated: RANGE_HOURLYRATE set to [30, 100]
Config updated: RANGE_JOBINVOLVEMENT set to [1, 4]
Config updated: RANGE_JOBLEVEL set to [1, 5]
Config updated: RANGE_JOBSATISFACTION set to [1, 4]
Config updated: RANGE_MONTHLYINCOME set to [1009, 19999]
Config updated: RANGE_MONTHLYRATE set to [2094, 26999]
Config updated: RANGE_NUMCOMPANIESWORKED set to [0, 9]
Config updated: RANGE_PERCENTSALARYHIKE set to [11, 25]
Config updated: RANGE_PERFORMANCERATING set to [3, 4]
Config updated: RANGE_RELATIONSHIPSATISFACTION set to [1, 4]
Config updated: RANGE_STOCKOPTIONLEVEL set to [0, 3]
Config updated: RANGE_TOTALWORKINGYEARS set to [0, 40]
Config updated: RANGE_TRAININGTIMESLASTYEAR set to [0, 6]
Config updated: RANGE_WORKLIFEBALANCE set to [1, 4]
Config updated: RANGE_YEARSATCOMPANY set to [0, 40]
Config updated: RANGE_YEARSINCURRENTROLE set to [0, 18]
Config updated: RANGE_YE

# **Data Defense**

In [40]:
def check_data(data: pd.DataFrame, config: dict) -> None:
    """
    Check the data against the specifications in the config file.

    Parameters:
    -----------
    data (pd.DataFrame): The data to be checked.
    config (dict): The configuration file.

    Returns:
    --------
    None: The function will raise an assertion error if any of the checks fail.
    """
    # check data types
    assert set(data.columns) == set(config['ALL_COLS']), "Data columns don't match the all columns specified in the config file."
    assert set(data.select_dtypes(include=['object']).columns) == set(obj_cols), "Object columns don't match."
    assert set(data.select_dtypes(include=['int64', 'float64']).columns) == set(num_cols), "Numerical columns don't match."

    # check target column
    assert config['TARGET_COL'] in data.columns, "Target column specified in the config file is not present in the data."
    
    # check range of numeric columns
    for col in num_cols:
        assert data[col].between(config[f'RANGE_{col.upper()}'][0], config[f'RANGE_{col.upper()}'][1]).sum() == len(data), f"An error occurred in {col} range. Please check the data and config file."

In [41]:
# execute check data function
check_data(data=data, config=config)

# **Data Split**

In [42]:
def split_data(data: pd.DataFrame, config: dict) -> Tuple[pd.DataFrame, pd.DataFrame, pd.Series, pd.Series]:

    """

    Split the data into training and testing sets.

    Parameters:
    -----------
    data (pd.DataFrame): The data to be split.
    config (dict): The configuration file.

    Returns:
    --------
    X_train (pd.DataFrame): The training input features.
    X_test (pd.DataFrame): The testing input features.
    y_train (pd.Series): The training target variable.
    y_test (pd.Series): The testing target variable.

    """

    # split input and output
    X = data.drop(columns=[config['TARGET_COL']])
    y = data[config['TARGET_COL']]

    # split train and test
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=config['TEST_SIZE'], random_state=config['RANDOM_STATE'], stratify=y
    )
    
    print(f'Data split into training and testing sets successfully.')
    print(f'X_train shape: {X_train.shape}, y_train shape: {y_train.shape}')
    print(f'X_test shape: {X_test.shape}, y_test shape: {y_test.shape}')

    return X_train, X_test, y_train, y_test

In [43]:
X_train, X_test, y_train, y_test = split_data(data=data, config=config)

Data split into training and testing sets successfully.
X_train shape: (1176, 30), y_train shape: (1176,)
X_test shape: (294, 30), y_test shape: (294,)


In [44]:
# split cat and num cols
cat_cols_train = X_train.select_dtypes(include='object').columns.tolist()
num_cols_train = X_train.select_dtypes(include=['int64', 'float64']).columns.tolist()

config = update_config(key='CAT_COLS', value=cat_cols_train, config_path=PROJECT_ROOT / 'config' / 'config.yaml')
config = update_config(key='NUM_COLS', value=num_cols_train, config_path=PROJECT_ROOT / 'config' / 'config.yaml')

Config updated: CAT_COLS set to ['BusinessTravel', 'Department', 'Education', 'EducationField', 'Gender', 'JobRole', 'MaritalStatus', 'OverTime']
Config updated: NUM_COLS set to ['Age', 'DailyRate', 'DistanceFromHome', 'EnvironmentSatisfaction', 'HourlyRate', 'JobInvolvement', 'JobLevel', 'JobSatisfaction', 'MonthlyIncome', 'MonthlyRate', 'NumCompaniesWorked', 'PercentSalaryHike', 'PerformanceRating', 'RelationshipSatisfaction', 'StockOptionLevel', 'TotalWorkingYears', 'TrainingTimesLastYear', 'WorkLifeBalance', 'YearsAtCompany', 'YearsInCurrentRole', 'YearsSinceLastPromotion', 'YearsWithCurrManager']


In [45]:
# define a function to serialize any object
def serialize_object(path: str, obj: Any) -> None:
    """ 
    Serialize the object into specified path.

    Parameters:
    -----------
    path (str): The path to save the serialized object.
    obj (Any): The object to be serialized.

    Returns:
    --------
    None
    """
    # save the object to the specified path
    joblib.dump(value=obj, filename=path)
    print(f'Saving object. . . .')
    print(f'Object serialized successfully to: {path}')

In [46]:
# serialize the splitted data
path_splitted_data = PROJECT_ROOT / 'data' / 'interim' 
serialize_object(path=path_splitted_data / 'X_train.pkl', obj=X_train)
serialize_object(path=path_splitted_data / 'X_test.pkl', obj=X_test)
serialize_object(path=path_splitted_data / 'y_train.pkl', obj=y_train)
serialize_object(path=path_splitted_data / 'y_test.pkl', obj=y_test)

Saving object. . . .
Object serialized successfully to: /home/bagaskoroah/employee_attrition/data/interim/X_train.pkl
Saving object. . . .
Object serialized successfully to: /home/bagaskoroah/employee_attrition/data/interim/X_test.pkl
Saving object. . . .
Object serialized successfully to: /home/bagaskoroah/employee_attrition/data/interim/y_train.pkl
Saving object. . . .
Object serialized successfully to: /home/bagaskoroah/employee_attrition/data/interim/y_test.pkl


# **Update Config**

In [47]:
# update splitted data path to config
config = update_config(
    key='PATH_TRAIN_DATA',
    value=['X_train.pkl', 'y_train.pkl'],
    config_path=PROJECT_ROOT / 'config' / 'config.yaml'
)

config = update_config(
    key='PATH_TEST_DATA',
    value=[f'X_test.pkl', 'y_test.pkl'],
    config_path=PROJECT_ROOT / 'config' / 'config.yaml'
)

Config updated: PATH_TRAIN_DATA set to ['X_train.pkl', 'y_train.pkl']
Config updated: PATH_TEST_DATA set to ['X_test.pkl', 'y_test.pkl']


In [48]:
# sanity check on config yaml file
config

{'ALL_COLS': ['Age',
  'Attrition',
  'BusinessTravel',
  'DailyRate',
  'Department',
  'DistanceFromHome',
  'Education',
  'EducationField',
  'EnvironmentSatisfaction',
  'Gender',
  'HourlyRate',
  'JobInvolvement',
  'JobLevel',
  'JobRole',
  'JobSatisfaction',
  'MaritalStatus',
  'MonthlyIncome',
  'MonthlyRate',
  'NumCompaniesWorked',
  'OverTime',
  'PercentSalaryHike',
  'PerformanceRating',
  'RelationshipSatisfaction',
  'StockOptionLevel',
  'TotalWorkingYears',
  'TrainingTimesLastYear',
  'WorkLifeBalance',
  'YearsAtCompany',
  'YearsInCurrentRole',
  'YearsSinceLastPromotion',
  'YearsWithCurrManager'],
 'CAT_COLS': ['BusinessTravel',
  'Department',
  'Education',
  'EducationField',
  'Gender',
  'JobRole',
  'MaritalStatus',
  'OverTime'],
 'NUM_COLS': ['Age',
  'DailyRate',
  'DistanceFromHome',
  'EnvironmentSatisfaction',
  'HourlyRate',
  'JobInvolvement',
  'JobLevel',
  'JobSatisfaction',
  'MonthlyIncome',
  'MonthlyRate',
  'NumCompaniesWorked',
  'Percen